# Real-Load Test — Switzerland (1 000 pairs)

Simulates realistic API traffic against the full Switzerland-wide planner
(no geographic region filter — all Swiss stops with BPUIC prefix `85`).

**Pair distribution** mirrors how real users query a national transit planner:
- **50 %** hub ↔ hub (top-30 busy stops — major Swiss hubs, high contention)
- **25 %** hub ↔ secondary (top-30 → top 31–150)
- **15 %** secondary ↔ secondary
- **10 %** fully random from any stop in the network

Because popular hub pairs are overrepresented, the TTL cache absorbs many
duplicate calls — exactly as it would in production.

> **Note:** `prepare()` with `regions=()` loads the full Swiss timetable.
> Expect ~8 000+ stops and significantly more trips than the Lausanne-only test.
> First-time preparation takes several minutes.

**Workflow**
1. Run **Setup** once per kernel session.
2. Run **Prepare planner** (only needed once; skip if already prepared).
3. Run **Generate pairs** → **Benchmark** → **Results**.

## Setup

In [1]:
import importlib
import os
import sys

cwd = os.path.abspath(os.getcwd())
project_root = cwd if os.path.exists(os.path.join(cwd, "src")) else os.path.abspath(os.path.join(cwd, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import src.config.settings as _s
import src.data.csa_data_handler as _cdh
import src.models.delay_model as _dm
import src.models.delay_model_trainer as _dmt
import src.models.model_artifacts as _ma
import src.routing.robust_journey_planner as _rjp
import tests.test_robust_journey as _bench


def _prepared_summary(p):
    if p is None or not getattr(p, "prepared", False):
        return " (robust planner is not prepared yet)"
    days = getattr(p, "connections_by_day", None) or {}
    n_day_connections = sum(len(v) for v in days.values())
    n_stops = len(getattr(p, "stops", None) or [])
    n_trips = getattr(p, "n_trips", 0)
    has_delays = bool(getattr(p, "delay_lookup", None))
    return f" ({n_stops:,} stops, {n_trips:,} trips, {n_day_connections:,} day-connections, delay_lookup={has_delays})"


def reload_robust_code(preserve_prepared=True):
    global settings, get_settings, RobustJourneyPlanner
    global bench_robust_prepare, bench_robust_plan
    global _s, _cdh, _dm, _dmt, _ma, _rjp, _bench, robust_planner

    old_planner = globals().get("robust_planner")
    for module in (_s, _cdh, _dm, _dmt, _ma, _rjp, _bench):
        importlib.reload(module)

    get_settings = _s.get_settings
    RobustJourneyPlanner = _rjp.RobustJourneyPlanner
    bench_robust_prepare = _bench.bench_robust_prepare
    bench_robust_plan = _bench.bench_robust_plan

    try:
        settings = get_settings()
    except Exception:
        if old_planner is not None and hasattr(old_planner, "settings"):
            settings = old_planner.settings
        elif "settings" in globals():
            settings = globals()["settings"]
        else:
            raise

    new_planner = RobustJourneyPlanner(settings=settings)
    if preserve_prepared and old_planner is not None and getattr(old_planner, "prepared", False):
        for name, value in vars(old_planner).items():
            setattr(new_planner, name, value)
        new_planner.settings = settings
        if getattr(new_planner, "data_handler", None) is not None:
            try:
                new_planner.data_handler.__class__ = _cdh.CSADataHandler
            except TypeError:
                pass
        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; preserved prepared data" + _prepared_summary(robust_planner))
    else:
        robust_planner = new_planner
        print("Reloaded robust_journey_planner.py; created a fresh unprepared robust planner")
    return robust_planner


robust_planner = reload_robust_code(preserve_prepared=True)

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner


In [2]:
# Reload latest code while keeping prepared data in memory.
robust_planner = reload_robust_code()

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner


In [10]:
# regions=() means no geographic filter → all Swiss stops (BPUIC prefix 85)
REGION_UUIDS          = None
REGION_LABEL          = "Switzerland (all)"

TRAVEL_DATE           = "2026-05-27"
DEADLINE              = "18:00"
# Confidence sampled per query — range 0.5–0.9, skewed towards the lower end
CONFIDENCE_LEVELS  = [0.50, 0.60, 0.70, 0.75, 0.80, 0.90]
CONFIDENCE_WEIGHTS = [30,   25,   20,   15,    7,    3   ]
MAX_ROUTES            = 3
SEARCH_WINDOW_MINUTES = 300
N_PAIRS               = 1000

print(f"Region        : {REGION_LABEL}")
print(f"Regions       : {REGION_UUIDS!r}  (empty = all Switzerland)")
print(f"Travel date   : {TRAVEL_DATE}")
print(f"Deadline      : {DEADLINE}")
print(f"Confidence    : {CONFIDENCE_LEVELS}")
print(f"  weights     : {CONFIDENCE_WEIGHTS}")
print(f"Max routes    : {MAX_ROUTES}")
print(f"Search window : {SEARCH_WINDOW_MINUTES} min")
print(f"Pairs         : {N_PAIRS}")
print("OK")

Region        : Switzerland (all)
Regions       : None  (empty = all Switzerland)
Travel date   : 2026-05-27
Deadline      : 18:00
Confidence    : [0.5, 0.6, 0.7, 0.75, 0.8, 0.9]
  weights     : [30, 25, 20, 15, 7, 3]
Max routes    : 3
Search window : 300 min
Pairs         : 1000
OK


## Prepare planner

Skip this cell if the planner is already prepared (`robust_planner.prepared == True`).
Set `FORCE_PREPARE = True` to rebuild Trino tables and reload all data.

> First-time preparation for the full Swiss network typically takes **5–10 minutes**
> (Trino table build + fetch of ~5 M connection rows + in-memory materialization).

In [4]:
FORCE_PREPARE             = False
FORCE_REBUILD             = False
REBUILD_CSA_PREREQUISITES = False

if robust_planner.prepared and not FORCE_PREPARE:
    print("robust_planner already prepared; skipping. Set FORCE_PREPARE=True to run again.")
else:
    robust_planner = reload_robust_code(preserve_prepared=False)
    bench_robust_prepare(
        robust_planner,
        regions=REGION_UUIDS,           # () = all Switzerland
        force_rebuild=FORCE_REBUILD,
        rebuild_csa_prerequisites=REBUILD_CSA_PREREQUISITES,
        load_delay_lookup=True,
        travel_date=TRAVEL_DATE,
    )

Reloaded robust_journey_planner.py; created a fresh unprepared robust planner

  bench_robust_prepare
  LOAD DELAY LOOKUP
  delay_lookup.load               0.02s  (0 keys)

  delay model loaded  (7 quantile boosters)


/home/kuci/project/final/src/data/csa_data_handler.py:434: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, self.conn)



  BUILD CSA TABLES
  skipping  stop_times_seq (already exists)
  build_stop_times_seq            0.09s
  skipping  stop_times_trips_seq (already exists)
  build_stop_times_trips_seq      0.08s
  skipping  full_table_seq (already exists)
  build_full_table_seq            0.09s
  skipping  connections (already exists)
  build_connections               0.09s
--------------------------------------------
  total build_all                 0.35s

  FETCH DATA
  fetch_stops                     0.78s  (25752 rows)
  fetch_footpaths                 0.86s  (52104 rows)
  fetch_connections             276.20s  (14508919 rows)

  MATERIALIZE IN-MEMORY
  stops dict                      0.04s  (25752 stops)
  footpaths dict                  0.14s  (52104 edges)
  connections loop               83.09s  (14508919 rows)
  sort + columns                 72.00s  (1340906 trips)
  trip meta by idx                0.90s
--------------------------------------------
  TOTAL prepare()               442.45s

  

In [5]:
planner = robust_planner
print(f"prepared     = {planner.prepared}")
print(f"delay_lookup = {bool(planner.delay_lookup)}")
print(f"n_stops      = {len(planner.stops)}")
print(f"n_trips      = {planner.n_trips}")

prepared     = True
delay_lookup = False
n_stops      = 25752
n_trips      = 1340906


## Generate realistic test pairs

Stops are ranked by how often they appear across weekday connections (Mon–Fri).
On the full Swiss network the top hubs are the major intercity stations
(Zürich HB, Bern, Basel, Geneva, Lausanne, etc.).

| Tier | Definition | Role |
|------|-----------|------|
| Hub (top 30) | national interchange stations | 50 % hub↔hub + half of the 25 % hub↔mid |
| Mid (31–150) | regional stops | other half of hub↔mid + 15 % mid↔mid |
| Low (151+)   | local stops    | part of 10 % random |

Duplicated pairs arise naturally from the skewed distribution, matching
the real workload pattern where the cache absorbs repeated popular-route requests.

In [6]:
import random
import statistics
from collections import Counter

stop_freq = Counter()
for day in ["monday", "tuesday", "wednesday", "thursday", "friday"]:
    for conn in planner.connections_by_day[day]:
        stop_freq[conn[0]] += 1
        stop_freq[conn[1]] += 1

ranked   = [s for s, _ in stop_freq.most_common()]
n_all    = len(ranked)
top_tier = ranked[:30]
mid_tier = ranked[30:min(150, n_all)]

print(f"Ranked stops : {n_all} total")
print(f"  hub  (top 30)   : {len(top_tier)}")
print(f"  mid  (31-150)   : {len(mid_tier)}")
print(f"  low  (151+)     : {n_all - len(top_tier) - len(mid_tier)}")
print(f"\nTop-10 hubs by connection frequency:")
for rank, (stop_id, freq) in enumerate(stop_freq.most_common(10), 1):
    meta = planner.stop_metadata.get(stop_id, {})
    name = meta.get("stop_name") or str(stop_id)
    print(f"  {rank:>2}. {name:<30} ({freq:,} appearances)")

rng         = random.Random(42)
TEST_PAIRS  = []
TEST_CONFS  = []
tier_labels = []

while len(TEST_PAIRS) < N_PAIRS:
    r = rng.random()
    if r < 0.50:
        s, e  = rng.choice(top_tier), rng.choice(top_tier)
        label = "hub↔hub"
    elif r < 0.75:
        s = rng.choice(top_tier)
        e = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        if rng.random() < 0.5:
            s, e = e, s
        label = "hub↔mid"
    elif r < 0.90:
        s = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        e = rng.choice(mid_tier) if mid_tier else rng.choice(ranked)
        label = "mid↔mid"
    else:
        s, e  = rng.choice(ranked), rng.choice(ranked)
        label = "random"
    if s != e:
        TEST_PAIRS.append((s, e))
        TEST_CONFS.append(rng.choices(CONFIDENCE_LEVELS, weights=CONFIDENCE_WEIGHTS, k=1)[0])
        tier_labels.append(label)

n_unique = len(set(TEST_PAIRS))
print(f"\nGenerated {len(TEST_PAIRS)} pairs:")
for cat, cnt in Counter(tier_labels).most_common():
    print(f"  {cat:<20} {cnt:>4}  ({100 * cnt / N_PAIRS:.0f}%)")
print(f"\nUnique pairs : {n_unique}")
print(f"Duplicates   : {len(TEST_PAIRS) - n_unique}")

conf_dist = Counter(TEST_CONFS)
print(f"\nConfidence distribution:")
for lv in CONFIDENCE_LEVELS:
    cnt = conf_dist.get(lv, 0)
    bar = "█" * (cnt // 10)
    print(f"  q={lv:.2f}  {cnt:>4}  ({100 * cnt / N_PAIRS:.0f}%)  {bar}")

Ranked stops : 25557 total
  hub  (top 30)   : 30
  mid  (31-150)   : 120
  low  (151+)     : 25407

Top-10 hubs by connection frequency:
   1. Genève, gare Cornavin          (131,086 appearances)
   2. Genève, Bel-Air                (112,989 appearances)
   3. Bern, Bahnhof                  (99,124 appearances)
   4. Bern, Zytglogge                (77,703 appearances)
   5. Luzern, Kantonalbank           (72,392 appearances)
   6. Genève, Rive                   (70,800 appearances)
   7. Genève, Plainpalais            (69,842 appearances)
   8. Schreckfeld (Firstbahn)        (68,840 appearances)
   9. Bort (Firstbahn)               (68,840 appearances)
  10. Schwyz, Zentrum                (68,720 appearances)

Generated 1000 pairs:
  hub↔hub               482  (48%)
  hub↔mid               261  (26%)
  mid↔mid               150  (15%)
  random                107  (11%)

Unique pairs : 888
Duplicates   : 112

Confidence distribution:
  q=0.50   306  (31%)  █████████████████████████████

## Benchmark — 1 000 queries

In [11]:
import time

robust_planner._route_cache.clear()   # start cold; no pre-warmed cache entries

all_times  = []
all_routes = []
no_route   = 0

for idx, ((start_stop, end_stop), conf_q) in enumerate(zip(TEST_PAIRS, TEST_CONFS)):
    t0 = time.perf_counter()
    result = robust_planner.plan(
        start_stop_id=start_stop,
        end_stop_id=end_stop,
        travel_date=TRAVEL_DATE,
        arrival_deadline=DEADLINE,
        confidence_q=conf_q,
        max_routes=MAX_ROUTES,
        search_window_minutes=SEARCH_WINDOW_MINUTES,
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    all_times.append(elapsed_ms)
    n = len(result)
    all_routes.append(n)
    if n == 0:
        no_route += 1
    if (idx + 1) % 100 == 0:
        print(f"  [{idx + 1:>4}/{N_PAIRS}]  running avg: {statistics.mean(all_times):.1f} ms")

print(f"\nFinished {len(all_times)} queries.")

  plan                total=253.4ms  scanned=148592  routes=3
  plan                total=162.6ms  scanned=149532  routes=3
  plan                total=3840.0ms  scanned=1434744  routes=0
  plan                total=4794.2ms  scanned=1779380  routes=3
  plan                total=6163.8ms  scanned=1779380  routes=1
  plan                total=3444.9ms  scanned=1679955  routes=0
  plan                total=4174.5ms  scanned=1779380  routes=0
  plan                total=1637.2ms  scanned=1779380  routes=0
  plan                total=4329.3ms  scanned=1759654  routes=2
  plan                total=11472.6ms  scanned=1120219  routes=0
  plan                total=3676.5ms  scanned=1779380  routes=0
  plan                total=4410.0ms  scanned=1779380  routes=0
  plan                total=78.1ms  scanned=84864  routes=3
  plan                total=5904.1ms  scanned=1779380  routes=0
  plan                total=4143.0ms  scanned=1779380  routes=0
  plan                total=4929.7ms  scanned=1

## Results

In [12]:
import pandas as pd

def percentile(lst, p):
    k = max(0, min(int(len(lst) * p / 100), len(lst) - 1))
    return lst[k]

sorted_times = sorted(all_times)
n            = len(sorted_times)
cache_hits   = sum(1 for t in all_times if t < 1.0)
conf_dist    = Counter(TEST_CONFS)

settings_data = [
    ("Region",           REGION_LABEL),
    ("Travel date",      TRAVEL_DATE),
    ("Deadline",         DEADLINE),
    ("Confidence range", "0.50–0.90 (skewed low)"),
    ("Max routes",       str(MAX_ROUTES)),
    ("Search window",    f"{SEARCH_WINDOW_MINUTES} min"),
    ("Total pairs",      str(N_PAIRS)),
    ("Unique pairs",     str(len(set(TEST_PAIRS)))),
    ("Duplicates",       str(len(TEST_PAIRS) - len(set(TEST_PAIRS)))),
]

timing_data = [
    ("avg",    f"{statistics.mean(all_times):.2f} ms"),
    ("median", f"{percentile(sorted_times, 50):.2f} ms"),
    ("p75",    f"{percentile(sorted_times, 75):.2f} ms"),
    ("p95",    f"{percentile(sorted_times, 95):.2f} ms"),
    ("p99",    f"{percentile(sorted_times, 99):.2f} ms"),
    ("max",    f"{sorted_times[-1]:.2f} ms"),
    ("stdev",  f"{statistics.stdev(all_times):.2f} ms"),
]

quality_data = [
    ("avg routes",  f"{statistics.mean(all_routes):.2f}"),
    ("no-route %",  f"{100 * no_route / n:.1f}%"),
    ("cache hits",  f"{cache_hits}  ({100 * cache_hits / n:.0f}% of calls < 1 ms)"),
]

dist_data = [(cat, cnt, f"{100 * cnt / N_PAIRS:.0f}%")
             for cat, cnt in Counter(tier_labels).most_common()]

conf_data = [(f"q={lv:.2f}", conf_dist.get(lv, 0), f"{100 * conf_dist.get(lv, 0) / N_PAIRS:.0f}%")
             for lv in CONFIDENCE_LEVELS]

SEP = "=" * 44

print(SEP)
print("  TEST SETTINGS")
print(SEP)
for k, v in settings_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  TIMING RESULTS")
print(SEP)
for k, v in timing_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  ROUTE QUALITY")
print(SEP)
for k, v in quality_data:
    print(f"  {k:<22} {v}")

print()
print(SEP)
print("  PAIR DISTRIBUTION")
print(SEP)
for cat, cnt, pct in dist_data:
    print(f"  {cat:<22} {cnt:>4}  ({pct})")

print()
print(SEP)
print("  CONFIDENCE DISTRIBUTION")
print(SEP)
for label, cnt, pct in conf_data:
    bar = "█" * (cnt // 10)
    print(f"  {label}  {cnt:>4}  ({pct})  {bar}")

print()
df_settings = pd.DataFrame(settings_data, columns=["Setting",    "Value"]).set_index("Setting")
df_timing   = pd.DataFrame(timing_data,   columns=["Metric",     "Value"]).set_index("Metric")
df_quality  = pd.DataFrame(quality_data,  columns=["Metric",     "Value"]).set_index("Metric")
df_dist     = pd.DataFrame(dist_data,     columns=["Category",   "Count", "Share"]).set_index("Category")
df_conf     = pd.DataFrame(conf_data,     columns=["Confidence", "Count", "Share"]).set_index("Confidence")

display(df_settings)
display(df_timing)
display(df_quality)
display(df_dist)
display(df_conf)

  TEST SETTINGS
  Region                 Switzerland (all)
  Travel date            2026-05-27
  Deadline               18:00
  Confidence range       0.50–0.90 (skewed low)
  Max routes             3
  Search window          300 min
  Total pairs            1000
  Unique pairs           888
  Duplicates             112

  TIMING RESULTS
  avg                    3792.95 ms
  median                 3529.97 ms
  p75                    4494.62 ms
  p95                    13107.16 ms
  p99                    14025.11 ms
  max                    15690.33 ms
  stdev                  3411.10 ms

  ROUTE QUALITY
  avg routes             1.26
  no-route %             49.0%
  cache hits             5  (0% of calls < 1 ms)

  PAIR DISTRIBUTION
  hub↔hub                 482  (48%)
  hub↔mid                 261  (26%)
  mid↔mid                 150  (15%)
  random                  107  (11%)

  CONFIDENCE DISTRIBUTION
  q=0.50   306  (31%)  ██████████████████████████████
  q=0.60   246  (25%)  █████

,Value
Setting,
Region,Switzerland (all)
Travel date,2026-05-27
Deadline,18:00
Confidence range,0.50–0.90 (skewed low)
Max routes,3
Search window,300 min
Total pairs,1000
Unique pairs,888
Duplicates,112


,Value
Metric,
avg,3792.95 ms
median,3529.97 ms
p75,4494.62 ms
p95,13107.16 ms
p99,14025.11 ms
max,15690.33 ms
stdev,3411.10 ms


,Value
Metric,
avg routes,1.26
no-route %,49.0%
cache hits,5 (0% of calls < 1 ms)


,Count,Share
Category,,
hub↔hub,482,48%
hub↔mid,261,26%
mid↔mid,150,15%
random,107,11%


,Count,Share
Confidence,,
q=0.50,306,31%
q=0.60,246,25%
q=0.70,193,19%
q=0.75,162,16%
q=0.80,68,7%
q=0.90,25,2%
